In [1]:
#ADDED THE MULTITHREADING FUNCTIONALITY, PROCESSING SPEED HAS IMPROVED CONSIDERABLY, 100 PAGES (~2500 LISTINGS) SCRAPED IN AROUND 25 MINS

#ADDED THE MULTITHREADING FUNCTIONALITY, PROCESSING SPEED HAS IMPROVED CONSIDERABLY, 100 PAGES (~2500 LISTINGS) SCRAPED IN AROUND 25 MINS


import requests
from bs4 import BeautifulSoup
import re
import time
import pandas as pd
import json
from datetime import datetime, timedelta
import concurrent.futures
import os
import io
import logging
import concurrent.futures
from urllib.parse import urlparse
from bs4 import BeautifulSoup
import boto3  # pip install boto3
from botocore.config import Config


# making the scraper behave like a standard web browser. 
BASE_URL = "https://www.zameen.com"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Accept-Encoding": "gzip, deflate",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "DNT": "1",
    "Connection": "close",
    "Upgrade-Insecure-Requests": "1"
}

# Functions
def convert_price(price_str):
    price_str = str(price_str).replace(",", "").strip()
    if not price_str: return 0.0
    if "Crore" in price_str: return round(float(price_str.replace('Crore', '').strip()) * 10_000_000)
    elif "Lakh" in price_str: return round(float(price_str.replace('Lakh', '').strip()) * 100_000)
    elif "Million" in price_str: return round(float(price_str.replace('Million', '').strip()) * 1_000_000)
    elif "Arab" in price_str: return round(float(price_str.replace('Arab', '').strip()) * 1_000_000_000)
    elif "Thousand" in price_str: return round(float(price_str.replace('Thousand', '').strip()) * 1_000)
    else:
        try: return round(float(re.sub(r'[^\d.]', '', price_str)))
        except ValueError: return 0.0

def convert_size(size_str):
    size_str = str(size_str).replace(",", "").strip()
    if not size_str:
        return 0.0

    try:
        if "Marla" in size_str:
            return round(float(size_str.replace('Marla', '').strip()) * 225, 2)
        elif "Kanal" in size_str:
            return round(float(size_str.replace('Kanal', '').strip()) * 4500, 2)  # 1 Kanal = 20 Marla = 4500 sq ft
        elif "Sq. Yd." in size_str:
            return round(float(size_str.replace('Sq. Yd.', '').strip()) * 9, 2)  # 1 sq yd = 9 sq ft
        else:
            return round(float(size_str), 2)
    except ValueError:
        return 0.0


def convert_relative_date_to_absolute(relative_date_str):
    if not relative_date_str: return None
    relative_date_str = relative_date_str.lower().strip()
    now = datetime.now()
    if "just now" in relative_date_str or "few seconds ago" in relative_date_str: return now.strftime('%Y-%m-%d')
    elif "yesterday" in relative_date_str: return (now - timedelta(days=1)).strftime('%Y-%m-%d')
    parts = relative_date_str.split()
    if len(parts) < 2: return None
    try: value = int(parts[0]); unit = parts[1]
    except ValueError: return None
    delta = None
    if "minute" in unit: delta = timedelta(minutes=value)
    elif "hour" in unit: delta = timedelta(hours=value)
    elif "day" in unit: delta = timedelta(days=value)
    elif "week" in unit: delta = timedelta(weeks=value)
    elif "month" in unit: delta = timedelta(days=value * 30)
    elif "year" in unit: delta = timedelta(days=value * 365)
    if delta: return (now - delta).strftime('%Y-%m-%d')
    return None



In [2]:
#scraping listing links from the search results page
def get_listing_links(page, target_city_slug='Lahore-1'):
    """
    Extracts unique property detail page URLs from a Zameen.com search result page.
    Robust handling: detect 404 directly instead of relying on e.response.
    """
    search_url = f'https://www.zameen.com/Homes/{target_city_slug}-{page}.html'
    print(f"Fetching page {page}: {search_url}")

    try:
        response = requests.get(search_url, headers=headers, timeout=10)

        # ✅ Check 404 explicitly
        if response.status_code == 404:
            print(f"  Page {page} not found (404).")
            return None

        if response.status_code != 200:
            print(f"  Non-200 response ({response.status_code}) for {search_url}")
            return []

        # Only call raise_for_status for non-404 errors we didn’t catch
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')

        links = []
        for a_tag in soup.find_all('a', attrs={'aria-label': 'Listing link'},
                                   href=re.compile(r'/Property/.*\.html')):
            href = a_tag['href']
            full_url = BASE_URL + href
            if full_url not in links:
                links.append(full_url)

        if not links:
            print(f"  No new listing links found on page {page}. "
                  f"This might indicate end of results or a URL change.")
        return links

    except requests.exceptions.RequestException as e:
        # Handles timeouts, DNS errors, etc.
        print(f"Error fetching listing page {search_url}: {e}")
        return []



In [3]:
from dotenv import load_dotenv



load_dotenv()
# ----- CONFIG (set these appropriately) -----
B2_ENDPOINT = 'https://s3.us-east-005.backblazeb2.com' # change if needed
B2_ACCESS_KEY = os.getenv('B2_ACCESS_KEY', 'YOUR_KEY')
B2_SECRET_KEY = os.getenv('B2_SECRET_KEY', 'YOUR_SECRET')
B2_BUCKET = os.getenv('B2_BUCKET', 'your-bucket-name')
# PUBLIC URL template - adjust if you use custom domain or different endpoint style
# Example: https://{bucket}.s3.us-west-002.backblazeb2.com/{key}
PUBLIC_URL_TEMPLATE = os.getenv('B2_PUBLIC_URL_TEMPLATE',
                                f"https://{B2_BUCKET}.{urlparse(B2_ENDPOINT).hostname}/{{key}}")

print(f"Using Backblaze B2 endpoint: {B2_ENDPOINT}, bucket: {B2_BUCKET}")


# HTTP headers used while scraping
headers = {
    "User-Agent": "Mozilla/5.0 (compatible; PropertyScraper/1.0; +https://example.com/bot)"
}
#scraping related to single property detail page
# ---------- IMAGE HANDLER (separate function as requested) ----------
def upload_images_to_backblaze(image_urls, ad_id,
                               endpoint=B2_ENDPOINT,
                               access_key=B2_ACCESS_KEY,
                               secret_key=B2_SECRET_KEY,
                               bucket=B2_BUCKET,
                               public_url_template=PUBLIC_URL_TEMPLATE,
                               max_images=6,  # limit to first 6 images
                               retry=2):
    """
    Downloads up to 6 images from image_urls, uploads them to Backblaze (S3-compatible),
    and returns list of public URLs.
    """
    if not image_urls:
        return []

    session = boto3.session.Session()
    s3_client = session.client(
        's3',
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
        endpoint_url=endpoint,
        config=Config(signature_version='s3v4'),
        region_name=None
    )

    uploaded_urls = []
    for idx, url in enumerate(image_urls[:max_images], start=1):  # process only first 6
        try:
            resp = requests.get(url, headers=headers, stream=True, timeout=20)
            resp.raise_for_status()
            content_type = resp.headers.get('Content-Type', 'image/jpeg')

            parsed = urlparse(url)
            basename = os.path.basename(parsed.path)
            if not basename or '.' not in basename:
                ext = content_type.split('/')[-1] if '/' in content_type else 'jpg'
                basename = f"{idx}.{ext}"
            key = f"zameen/{ad_id}/{basename}"

            file_obj = io.BytesIO(resp.content)
            extra_args = {'ContentType': content_type}
            try:
                s3_client.upload_fileobj(file_obj, bucket, key, ExtraArgs=extra_args)
            except Exception as e:
                file_obj.seek(0)
                s3_client.upload_fileobj(file_obj, bucket, key)

            public_url = public_url_template.format(key=key)
            uploaded_urls.append(public_url)
            time.sleep(0.1)
        except Exception as e:
            uploaded_urls.append(None)
    return uploaded_urls

# ---------- IMAGE URL extractor ----------
def get_image_urls_from_soup(soup, base_url=None):
    """
    Collect image URLs from the page. Tries multiple heuristics:
    - finds gallery images (img tags under gallery/photo section)
    - finds large images (width/height or file path contains 'uploads'/'images' or 'cdn')
    - returns unique list preserving order
    """
    candidates = []
    # typical gallery images: find 'Photos' section then imgs
    # heuristic 1: images inside the main content
    for img in soup.select("div[id*='gallery'] img, div[class*='gallery'] img, .property-gallery img, .photos img"):
        src = img.get('data-src') or img.get('data-lazy') or img.get('src') or img.get('data-original')
        if src:
            candidates.append(src)

    # heuristic 2: any large images on page (filter out tiny icons)
    for img in soup.find_all('img'):
        src = img.get('data-src') or img.get('src') or img.get('data-original')
        if not src:
            continue
        # ignore icons / trackers: small files, base64, or images with 'sprite' or 'logo'
        if 'logo' in src or 'sprite' in src or 'icon' in src:
            continue
        # ensure http(s)
        if src.startswith('//'):
            src = 'https:' + src
        if src.startswith('/'):
            if base_url:
                src = base_url.rstrip('/') + src
            else:
                # cannot resolve relative without base, skip
                pass
        candidates.append(src)

    # cleaning and dedupe while preserving order
    seen = set()
    out = []
    for u in candidates:
        if not u: continue
        # remove thumbnail params if present (common pattern) - keep original file path
        u_clean = re.sub(r'(_thumb|_small|_x?\d+px).*$', '', u)
        if u_clean not in seen:
            seen.add(u_clean)
            out.append(u)
    return out

# ---------- AMENITIES & NEARBY parser ----------
def parse_amenities_and_nearby(soup, property_id):
    """
    Robust amenity extractor (only amenities; nearby is ignored).
    Heuristics:
     - look for headings containing 'amenit', 'feature', 'facility', 'facilit'
       and collect following siblings until the next heading
     - look for containers with class/id matching amenity-related keywords
     - fall back to scanning common element types (li, td, div.item, span)
     - also check title/alt/data-title attributes on icons/images
    Returns: list of amenity strings (deduped, order-preserving)
    """
    results = []
    fetched_at = datetime.utcnow().isoformat()

    def clean_text(t):
        if not t:
            return None
        # remove extraneous whitespace and common noise
        t = re.sub(r'\s+', ' ', t).strip()
        t = re.sub(r'^(show more[:\-]?\s*)', '', t, flags=re.I)
        t = re.sub(r'\(.*?\)$', '', t).strip()  # drop trailing parenthetical notes
        return t if t else None

    # 1) Headings-based extraction (walk siblings until next heading)
    heading_pattern = re.compile(r'(amenit|feature|facility|facilit|property[-\s]?features)', re.I)
    headings = soup.find_all(lambda tag: tag.name in ['h1','h2','h3','h4','h5'] and tag.get_text() and heading_pattern.search(tag.get_text()))
    for heading in headings:
        # look through siblings under this section
        current = heading
        section_nodes = []
        while current:
            # stop at the next heading of similar level or higher
            if current is not heading and current.name in ['h1','h2','h3','h4','h5']:
                break
            # collect useful blocks
            for tagname in ('ul','div','table','section'):
                for block in current.find_all(tagname, recursive=False):
                    section_nodes.append(block)
            # also consider immediate lists or grids within the subtree
            for li in current.find_all('li', recursive=True):
                section_nodes.append(li)
            # move to next sibling
            current = current.find_next_sibling()
        # extract texts from found nodes
        for node in section_nodes:
            # common elements that hold amenity text
            for candidate in node.find_all(['li','span','p','div','td','a'], recursive=True):
                text = candidate.get_text(separator=' ', strip=True)
                t = clean_text(text)
                if t:
                    results.append(t)
            # also check for images/icons with alt/title
            for ico in node.find_all(['img','i','svg'], recursive=True):
                alt = ico.get('alt') or ico.get('title') or ico.get('data-title')
                t = clean_text(alt)
                if t:
                    results.append(t)

    # 2) Container class/id based extraction (fallback)
    if not results:
        container_pattern = re.compile(r'(amenit|facility|feature|facilit|property[-_]?features|features-list|amenities-list)', re.I)
        containers = soup.find_all(attrs={'class': container_pattern}) + soup.find_all(attrs={'id': container_pattern})
        # unique containers preserving order
        seen_containers = []
        for c in containers:
            if c in seen_containers:
                continue
            seen_containers.append(c)
            # find likely amenity nodes inside
            for candidate in c.find_all(['li','div','span','p','td','a'], recursive=True):
                text = candidate.get_text(separator=' ', strip=True)
                t = clean_text(text)
                if t:
                    results.append(t)
            for ico in c.find_all(['img','i','svg'], recursive=True):
                alt = ico.get('alt') or ico.get('title') or ico.get('data-title')
                t = clean_text(alt)
                if t:
                    results.append(t)

    # 3) Wide fallback: look for common patterns across the page (small, but useful)
    if not results:
        # sometimes amenities are a grid of icons with labels inside divs with role/aria attributes
        for candidate in soup.find_all(['li','div','span','p','td','a'], recursive=True):
            # skip very long paragraphs
            txt = candidate.get_text(separator=' ', strip=True)
            if not txt or len(txt) > 150:
                continue
            # heuristics: short text, likely amenity-like (one or two words or contains colon)
            if re.search(r'(^[A-Za-z0-9 ]{2,40}$)|:', txt):
                t = clean_text(txt)
                if t:
                    results.append(t)
            # check attributes on the element
            alt = candidate.get('title') or candidate.get('data-title') or candidate.get('aria-label')
            if alt:
                t = clean_text(alt)
                if t:
                    results.append(t)

    # 4) Extract from icon/images globally (alt/title)
    for ico in soup.find_all(['img','i','svg'], recursive=True):
        alt = ico.get('alt') or ico.get('title') or ico.get('data-title')
        t = clean_text(alt)
        if t:
            results.append(t)

    # Final cleanup & dedupe while preserving order, and filter out junk
    seen = set()
    clean_results = []
    for r in results:
        if not r:
            continue
        # filter out generic junk like 'View Details' or 'More'
        if re.match(r'^(view|more|details|show|expand)\b', r, re.I):
            continue
        # normalize spacing and punctuation
        r_norm = re.sub(r'\s+', ' ', r).strip(' -:•\t\n\r ')
        if not r_norm:
            continue
        if r_norm in seen:
            continue
        seen.add(r_norm)
        clean_results.append(r_norm)

    # Build final amenity dict-like entries if you ever need timestamps etc.
    # But per your change request, return plain list of amenity names.
    return clean_results

def get_detail_page_data_from_dataLayer(detail_url):
    """
    Re-used from your original: Extracts dataLayer object from a detail page.
    """
    try:
        response = requests.get(detail_url, headers=headers, timeout=10)
        response.raise_for_status()

        dataLayer_pattern = re.compile(r"window\['dataLayer'\]\.push\((\{.*?\})\);", re.DOTALL)
        match = dataLayer_pattern.search(response.text)

        if match:
            json_str = match.group(1)
            try:
                data = json.loads(json_str)
                return data
            except json.JSONDecodeError as e:
                print("")
    except Exception as e:
        print("")
    
    return None

def parse_html_details(soup, data_layer_data, detail_url=None):
    """
    Extends your previous parse_html_details to also pull title, description, city.
    """
    details = {
        'Ad_ID': None, 'Price': None, 'Area': None, 'Bedrooms': None, 'Bathrooms': None,
        'Latitude': None, 'Longitude': None, 'Property_Type': None,
        'Location_Detail': None, 'Date_Added': None,
        'Title': None, 'Description': None, 'City': None
    }

    # Ad ID from dataLayer (NEW) - robust checks
    if data_layer_data and data_layer_data.get('ad_id'):
        details['Ad_ID'] = data_layer_data['ad_id']

    # Price
    price_tag = soup.find('span', attrs={'aria-label': 'Price'})
    if not price_tag:
        # fallback: look for price text
        price_tag = soup.find(text=re.compile(r'PKR|Rs\.|Rupee', re.I))
    price_raw = price_tag.text.strip() if getattr(price_tag, 'text', None) else (price_tag.strip() if isinstance(price_tag, str) else None)
    details['Price'] = convert_price(price_raw)

    # Bedrooms
    bedrooms_tag = soup.find('span', attrs={'aria-label': 'Beds'}) or soup.find(text=re.compile(r'\d+\s*Beds?', re.I))
    bedrooms_raw = bedrooms_tag.text.strip() if getattr(bedrooms_tag, 'text', None) else (bedrooms_tag.strip() if isinstance(bedrooms_tag, str) else None)
    if bedrooms_raw:
        beds_match = re.search(r'(\d+)', bedrooms_raw)
        details['Bedrooms'] = int(beds_match.group(1)) if beds_match else None

    # Bathrooms
    bathrooms_tag = soup.find('span', attrs={'aria-label': 'Baths'}) or soup.find(text=re.compile(r'\d+\s*Baths?', re.I))
    bathrooms_raw = bathrooms_tag.text.strip() if getattr(bathrooms_tag, 'text', None) else (bathrooms_tag.strip() if isinstance(bathrooms_tag, str) else None)
    if bathrooms_raw:
        baths_match = re.search(r'(\d+)', bathrooms_raw)
        details['Bathrooms'] = int(baths_match.group(1)) if baths_match else None

    # Area
    area_tag = soup.find('span', attrs={'aria-label': 'Area'}) or soup.find(text=re.compile(r'(Kanal|Marla|Sq\.)', re.I))
    area_raw = area_tag.text.strip() if getattr(area_tag, 'text', None) else (area_tag.strip() if isinstance(area_tag, str) else None)
    details['Area'] = convert_size(area_raw)

    # Property Type
    property_type_tag = soup.find('span', attrs={'aria-label': 'Type'}) or soup.find(text=re.compile(r'\bType\b', re.I))
    details['Property_Type'] = property_type_tag.text.strip() if getattr(property_type_tag, 'text', None) else None

    # Date Added
    creation_date_tag = soup.find('span', attrs={'aria-label': 'Creation date'}) or soup.find(text=re.compile(r'Added', re.I))
    creation_date_raw = creation_date_tag.text.strip() if getattr(creation_date_tag, 'text', None) else (creation_date_tag.strip() if isinstance(creation_date_tag, str) else None)
    details['Date_Added'] = convert_relative_date_to_absolute(creation_date_raw)

    # Title
    # Zameen usually shows H1 or a line with property title
    title_tag = soup.find(['h1', 'h2'], text=True)
    if title_tag and title_tag.get_text(strip=True):
        details['Title'] = title_tag.get_text(strip=True)
    else:
        # fallback from <title> tag
        title_tag = soup.find('title')
        if title_tag:
            details['Title'] = title_tag.get_text(strip=True)

    # Description - find heading "Description" then text following
    desc_heading = soup.find(lambda tag: tag.name in ['h2', 'h3', 'h4'] and 'description' in (tag.get_text() or '').lower())
    description = None
    if desc_heading:
        # gather subsequent sibling paragraphs until next heading
        parts = []
        sibling = desc_heading.find_next_sibling()
        while sibling and sibling.name not in ['h2', 'h3', 'h4']:
            parts.append(sibling.get_text(separator=' ', strip=True))
            sibling = sibling.find_next_sibling()
        description = ' '.join([p for p in parts if p]).strip()
    if not description:
        # fallback search for a description-like block
        desc_div = soup.find('div', attrs={'class': re.compile(r'description|desc', re.I)})
        if desc_div:
            description = desc_div.get_text(separator=' ', strip=True)
    details['Description'] = description

    # Latitude, Longitude, Location_Detail and City from dataLayer if present
    if data_layer_data:
        try:
            details['Latitude'] = float(data_layer_data.get('latitude')) if data_layer_data.get('latitude') is not None else None
            details['Longitude'] = float(data_layer_data.get('longitude')) if data_layer_data.get('longitude') is not None else None
        except:
            details['Latitude'], details['Longitude'] = None, None

        loc_components = []
        if data_layer_data.get('loc_neighbourhood_name'):
            loc_components.append(data_layer_data['loc_neighbourhood_name'])
        if data_layer_data.get('loc_name') and data_layer_data.get('loc_name') != data_layer_data.get('loc_neighbourhood_name'):
            loc_components.append(data_layer_data['loc_name'])
        if data_layer_data.get('loc_city_name') and data_layer_data.get('loc_city_name') not in loc_components:
            loc_components.append(data_layer_data['loc_city_name'])
            details['City'] = data_layer_data.get('loc_city_name')
        details['Location_Detail'] = ', '.join(loc_components).strip() if loc_components else None

    # fallback: parse location text from page
    if not details['Location_Detail']:
        loc_tag = soup.find(lambda tag: tag.name in ['p', 'div', 'span'] and re.search(r'\b[A-Za-z]+,\s*[A-Za-z\- ]+,\s*[A-Za-z ]+', (tag.get_text() or '')))
        if loc_tag:
            details['Location_Detail'] = loc_tag.get_text(strip=True)

    return details

# Helper function for Multithreading
def process_single_listing(url, upload_images=True, max_images_per_listing=6):
    """
    Processes a single property detail URL (updated):
     - parses dataLayer
     - parses HTML fields (title, desc, etc)
     - extracts image urls, uploads to Backblaze, returns uploaded URLs
     - extracts AMENITIES (only names) and APPENDS them into property Description
    NOTE: This version no longer returns a separate amenities list (returns empty list
    as the second return value for backward compatibility).
    """
    try:
        data_layer_data = get_detail_page_data_from_dataLayer(url)

        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        details = parse_html_details(soup, data_layer_data, detail_url=url)
        details['URL'] = url

        # City fallback
        if not details.get('City') and details.get('Location_Detail'):
            parts = [p.strip() for p in details['Location_Detail'].split(',')]
            if parts:
                details['City'] = parts[-1]

        # IMAGE WORKFLOW
        image_urls = get_image_urls_from_soup(soup, base_url=url)
        if image_urls:
            if upload_images:
                uploaded_urls = upload_images_to_backblaze(
                    image_urls,
                    details.get('Ad_ID') or 'unknown',
                    max_images=max_images_per_listing
                )
            else:
                uploaded_urls = image_urls
        else:
            uploaded_urls = []

        details['Image_URLs'] = uploaded_urls
        details['Image_URLs_csv'] = ','.join([u for u in uploaded_urls if u])

        # AMENITIES: get names only (no nearby). Append into Description.
        amenity_names = parse_amenities_and_nearby(soup, details.get('Ad_ID'))

        if amenity_names:
            existing_desc = details.get('Description') or ''
            if existing_desc:
                details['Description'] = f"{existing_desc}\n\nAmenities: {', '.join(amenity_names)}"
            else:
                details['Description'] = f"Amenities: {', '.join(amenity_names)}"

        # Return details and an empty amenities list for compatibility
        return details

    except Exception as e:
        print(f"[process_single_listing] Error processing {url}: {e}")
        return None


def get_detail_page_url_for_city(city_slug = 'Lahore-1', max_pages=100):
    all_detail_urls = []
    target_city_slug = city_slug

    for page in range(1, max_pages + 1):
        listing_links = get_listing_links(page, target_city_slug)
        if listing_links is None:
            print(f"Stopping pagination for {city_slug} at page {page} due to 404 or no more pages.")
            break
        all_detail_urls.extend(listing_links)
        time.sleep(2) 

    print(f"\nCollected {len(all_detail_urls)} unique detail page URLs.")
    return all_detail_urls

def extract_details_for_given_urls(all_detail_urls, max_workers=8):
    listings_results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_url = {executor.submit(process_single_listing, url): url for url in all_detail_urls}
        for future in concurrent.futures.as_completed(future_to_url):
            url = future_to_url[future]
            try:
                listing_dict = future.result()
                if listing_dict:
                    listings_results.append(listing_dict)
            except Exception as exc:
                # append placeholder to stay consistent
                listings_results.append({
                    'URL': url, 'Ad_ID': None, 'Price': None, 'Area': None,
                    'Bedrooms': None, 'Bathrooms': None, 'Latitude': None,
                    'Longitude': None, 'Property_Type': None,
                    'Location_Detail': None, 'Date_Added': None, 'Title': None,
                    'Description': None, 'City': None, 'Image_URLs': [], 'Image_URLs_csv': ''
                })

    print(listings_results)  # Make sure this is outside the loop and try block
    return listings_results


Using Backblaze B2 endpoint: https://s3.us-east-005.backblazeb2.com, bucket: PropPal


In [4]:



# Main Execution Function
def main():
    
    city_code_file = "city_codes.json"
    details_url_file = "detail_urls.json"
    results_file = "zameen_listing_results.json"
    all_results = []
    all_detail_urls = []

    #load if there are previous results
    try:
        with open(results_file, 'r', encoding='utf-8') as f:
            all_results = json.load(f)    
    
    except Exception as e:
        print("file not found: ", e)
    
    #load if there are previously extracted details page urls
    try:
        with open(details_url_file, 'r', encoding='utf-8') as f:
            all_detail_urls = json.load(f)    
    
    except Exception as e:
        print("file not found: ", e)   


    with open(city_code_file, 'r', encoding='utf-8') as f:
        city_slugs = json.load(f)
    
    with open(details_url_file, 'r', encoding='utf-8') as f:
        all_detail_urls = json.load(f)


    city_slugs_copy = city_slugs.copy()
    for city_slug in city_slugs:
        print(f"\nProcessing city: {city_slug}")
        cities_detail_urls = get_detail_page_url_for_city(city_slug=city_slug, max_pages=1500)
        all_detail_urls.extend(cities_detail_urls)

        #updating detail url file
        json.dump(all_detail_urls, open(details_url_file, 'w', encoding='utf-8'), indent=2)

        #updating city_codes.json to remove processed city
        city_slugs_copy.remove(city_slug)
        json.dump(city_slugs_copy, open(city_code_file, 'w', encoding='utf-8'), indent=2, ensure_ascii=False)
        time.sleep(5)



    all_detail_url_copy = all_detail_urls.copy()
    for i in range(int(len(all_detail_urls)/48 +1)):
        print(f"Progress: {i*48}/{len(all_detail_urls)} detail pages processed.")
        city_results = extract_details_for_given_urls(all_detail_urls[48*(i):48*(i+1)])
        print(city_results)
        
        all_results.extend(city_results)
        json.dump(all_results, open(results_file, 'w', encoding='utf-8'), indent=2, ensure_ascii=False)


        df = pd.DataFrame(all_results)
        json.dump(all_results, open('all_results_backup.json', 'w', encoding='utf-8'), indent=2, ensure_ascii=False)

        desired_columns = [
            'Ad_ID', 'Title: ', 'Description', 'Property_Type', 'Location_Detail', 'Price', 'Area', 'City',
            'Bedrooms', 'Bathrooms', 'Latitude', 'Longitude', 'Date_Added', 'Image_URLs', 'URL'
        ]
        for col in desired_columns:
            if col not in df.columns:
                df[col] = None
        df = df[desired_columns]

        if 'URL' in df.columns:
            df = df.drop(columns=['URL'])

        output_filename = 'zameen_listings_final_All.csv'
        df.to_csv(output_filename, index=False, encoding='utf-8')
        print(f"\n✅ Data saved to '{output_filename}'.")

        #removing urls already processed
        all_detail_url_copy = all_detail_urls[48*(i+1):]
        json.dump(all_detail_url_copy, open(details_url_file, 'w', encoding='utf-8'), indent=2)



if __name__ == "__main__":
    
    # city_results = extract_details_for_given_urls(["https://www.zameen.com/Property/abbottabad_nathia_gali_in_nathia_gali_flat_for_sale_sized_1008_square_feet-53094420-1248-1.html"])

    # print(city_results)
    main()

Progress: 0/0 detail pages processed.
[]
[]

✅ Data saved to 'zameen_listings_final_All.csv'.
